# Diffusion Concepts

**Module:** 17 — Image Generation

Noise schedules, latent diffusion, CFG, samplers, seeds, and practical tuning knobs.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Describe forward/reverse diffusion and latent diffusion
- Tune CFG, steps, and sampler with predicted side effects
- Use seeds for reproducibility and controlled variation
- Explain why distilled few-step models change latency


## Core Ideas — Forward and Reverse Processes

### Definition
The **forward process** destroys structure with noise; the **reverse process** is a learned denoiser reconstructing samples under conditioning.

### Why it matters
Every UI slider (steps, CFG, sampler) knobs the reverse process.

### How it works
Train ε_θ(x_t,t,cond); at inference start from noise and iterate a sampler to t=0, then decode if latent.

### Intuition
Developing a photo — successive baths reveal the image.

### Pitfalls
- Confusing training timesteps with UI steps across samplers
- Changing VAE mid-pipeline

### When to use
Anytime you debug quality/latency on diffusion systems.


```mermaid
flowchart LR
  X0[x0 clean] -->|noise| XT[xT noise]
  XT -->|denoise| X0h[x0 hat]
  COND[text / controls] --> DEN[Denoiser]
  DEN --> X0h
```

### Classifier-Free Guidance (CFG)
`ε = ε_uncond + scale * (ε_cond − ε_uncond)`

| CFG | Effect |
|-----|--------|
| 1–3 | Softer, more diverse |
| 5–8 | Common sweet spot |
| 12+ | Harsh, brittle artifacts |


In [ ]:
# Demo 1: CFG vector math
def cfg(eps_uncond, eps_cond, scale: float):
    return [u + scale * (c - u) for u, c in zip(eps_uncond, eps_cond)]

eps_u, eps_c = [0.1, -0.2], [0.5, 0.4]
for s in [1.0, 3.0, 7.5, 15.0]:
    print(s, [round(x, 3) for x in cfg(eps_u, eps_c, s)])


## Latent Diffusion

### Definition
Run diffusion in VAE latent space, then decode — huge savings vs pixel-space high-res diffusion.

### Why it matters
Why SD-class models are practical on consumer GPUs.

### How it works
Encode with VAE → denoise z_t → decode once at the end.

### Intuition
Compose as MIDI (latent) then render audio (decode).

### Pitfalls
- Mixing incompatible VAEs
- Decoding every intermediate step in prod

### When to use
Default for open high-res image generation.


In [ ]:
# Demo 2: latent vs pixel work
def relative_cost(h, w, scale=8):
    pix, lat = h * w * 3, (h // scale) * (w // scale) * 4
    return {"pixels": pix, "latents": lat, "ratio": round(pix / lat, 1)}

print((4, 128, 128), "latent shape example channels,h,w")
print(relative_cost(1024, 1024))


## Samplers, Seeds, Tuning

### Definition
Samplers integrate the reverse ODE/SDE; seeds fix RNG; tuning is single-variable ablation.

### Why it matters
Same checkpoint + different sampler can look like a different model.

### How it works
Lock sampler in prod; seed-lock for bugs; seed-sweep for exploration; distill for latency tiers.

### Intuition
GPS routes to the same city — some faster, some scenic.

### Pitfalls
- Copying step counts across samplers
- Changing model+CFG+seed at once in A/B

### When to use
After model and resolution are fixed.


### Sampler cheat sheet

| Family | Traits |
|--------|--------|
| Euler / Euler a | Simple; `a` adds stochasticity |
| DDIM | Deterministic-ish, fewer steps |
| DPM++ | Strong quality/speed for many SD models |
| Distilled (LCM/Turbo/Lightning) | 1–8 steps |

| Symptom | Try |
|---------|-----|
| Ignores prompt | Raise CFG slightly; simplify prompt |
| Fried / oversharp | Lower CFG; fewer LoRAs |
| Muddy | More steps; native resolution multiples |
| Almost right | Inpaint/img2img beats prompt thrash |


In [ ]:
# Demo 3: reproducible seed knobs
import random

def sample_knobs(seed: int) -> dict:
    rng = random.Random(seed)
    return {
        "palette": rng.choice(["teal", "amber", "mono"]),
        "camera": rng.choice(["35mm", "85mm", "drone"]),
        "weather": rng.choice(["clear", "overcast", "fog"]),
    }

print(sample_knobs(42)); print(sample_knobs(42)); print(sample_knobs(43))


In [ ]:
# Demo 4: tuning grid shrinkage
from itertools import product

grid = [
    {"steps": s, "cfg": c, "sampler": sam, "cost": s * (1 + 0.05 * c)}
    for s, c, sam in product([8, 20, 30], [3.5, 7.0, 11.0], ["euler", "dpmpp_2m"])
]
filtered = [g for g in grid if g["cfg"] == 7.0 and g["steps"] >= 20]
print("total", len(grid), "filtered", sorted(filtered, key=lambda x: x["cost"])[:3])


### Try it yourself — Diffusion tuning

1. Implement guided=uncond+s*(cond-uncond) for named feature channels.
2. Table step budgets for 2s vs 10s SLA at 40ms/step.
3. Document a locked production sampler config as JSON.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `noise schedule` | How β_t / σ_t evolves over timesteps |
| `CFG` | Classifier-free guidance scale |
| `sampler` | Algorithm advancing t → t−Δ |
| `distillation` | Few-step student of a many-step teacher |
| `ancestral sampler` | Adds noise each step |


### Workshop — Parameter journal — Diffusion Concepts

List every knob you touched. Predict the effect of changing one knob before changing it.


In [ ]:
# Workshop 1 — Diffusion Concepts
knobs = ['seed','guidance','steps','size','strength']
for k in knobs:
    print(f'{k}: value=?, hypothesis=?, observed=?')


### Workshop — Failure taxonomy — Diffusion Concepts

Classify bad outputs into: prompt, model, control, safety, or infra.


In [ ]:
# Workshop 2 — Diffusion Concepts
examples = ['ignored count','flicker','pose ignored','blocked','504']
for e in examples:
    print(e, '->', 'TODO-label')


### Workshop — Cost / latency card — Diffusion Concepts

Estimate unit cost for draft vs final tiers at a daily volume.


In [ ]:
# Workshop 3 — Diffusion Concepts
def monthly(qpd, price, days=30, retry=0.1):
    return round(qpd*days*(1+retry)*price, 2)
print('draft$', monthly(2000, 0.02))
print('final$', monthly(500, 0.08))


### Workshop — Eval golden item — Diffusion Concepts

Add one golden prompt/job with must-have attributes and reject criteria.


In [ ]:
# Workshop 4 — Diffusion Concepts
golden = {'id':'g1','prompt':'TODO','must_have':['subject','style'],'reject_if':['watermark']}
print(golden)


### Workshop — Safety + provenance — Diffusion Concepts

Write audit metadata: model hash, seed, policy version, credentials flag.


In [ ]:
# Workshop 5 — Diffusion Concepts
meta = {'model':'name@sha256:...','seed':0,'policy_version':'2026.04','credentials':True}
assert 'model' in meta
print(meta)


### Workshop — Ablation plan — Diffusion Concepts

Design a 4-run ablation changing only one variable each time.


In [ ]:
# Workshop 6 — Diffusion Concepts
runs = [{'id':i,'change':c,'score':None} for i,c in enumerate(['baseline','guidance-2','steps+10','new_seed'],1)]
print(runs)


### Workshop — Interface sketch — Diffusion Concepts

Sketch provider-agnostic request/response dicts for this modality.


In [ ]:
# Workshop 7 — Diffusion Concepts
req = {'prompt':'...','size':'...','seed':1}
resp = {'status':'ok','asset_uri':'...','model':'...','seed':1}
print(req); print(resp)


### Workshop — Self-check — Diffusion Concepts

Run this checklist printer and fill it honestly before moving on.


In [ ]:
# Workshop 8 — Diffusion Concepts
for i,x in enumerate(['restated objectives','ran demos','named pitfall','named metric'],1):
    print(f'{i}. [ ] {x}')


## Key Takeaways

- Quality knobs are mostly reverse-process controls
- Latent diffusion = VAE compression + denoise in z
- CFG and sampler interact — change one variable at a time
- Seeds make bugs reproducible; sweeps explore diversity
